# Assumption: the data from all the treated reservoirs is available up to the same date

### Import modules and data

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import numpy as np
from sklearn.metrics.pairwise import haversine_distances

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.transferring.predicting import predict_one_year, predict_one_year_sarima

critical_threshold = 0.075
worrying_threshold = 0.15
could_give_if_critical = 0.25
could_give_if_worrying = 0.35
able_to_donate = 0.6
cost_threshold = 1.5
epsilon = 1e-6

c:\Users\usuario\Desktop\Extra\Reservoir\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [3]:
water_path = PATHS['definitive_notebooks'] / 'water_definitive.parquet'
water_pd = pd.read_parquet(water_path)
water_pd.head()

,date,storage,id,storage_imputed
0,1988-01-05,0,3,0
1,1988-01-12,0,3,0
2,1988-01-19,0,3,0
3,1988-01-26,0,3,0
4,1988-02-02,0,3,0


In [4]:
reservoirs_path = PATHS['definitive_notebooks'] / 'reservoirs_merged.parquet'
reservoirs_pd = pd.read_parquet(reservoirs_path)
reservoirs_pd.head()

,id,scope,name,capacity,electric_flag,longitude,latitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,3,guadalquivir,fernandina,247.0,0,38.179646,-3.570224,guadalquivir,rio guarrizas,None,None,None,jaen,andalucia,presa fabrica gravedad (hormigon vibrado),719.55,NaN,https://sig.mapama.gob.es/WebServices/clientew...
1,5,guadalquivir,puebla cazalla,87.0,0,37.129772,-5.243309,guadalquivir,rio corbones,None,None,None,sevilla,andalucia,presa fabrica gravedad (hormigon compactado),218.25,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,9,cuenca mediterranea andaluza,vinuela,170.0,0,36.860400,-4.164435,cuencas mediterraneas andaluzas,rio guaro,https://www.google.com/search?kgmid=/g/120jr2sh,None,https://www.wikidata.org/wiki/Q5830515,malaga,andalucia,presa materiales sueltos pantalla hormigon,426.00,NaN,https://sig.mapama.gob.es/WebServices/clientew...
3,11,cuenca mediterranea andaluza,rules,111.0,0,36.860510,-3.495430,cuencas mediterraneas andaluzas,rio guadalfeo o rio cadiar,https://www.google.com/search?kgmid=/g/121vx8sj,None,https://www.wikidata.org/wiki/Q5369455,granada,andalucia,presa fabrica arco gravedad,426.00,NaN,https://sig.mapama.gob.es/WebServices/clientew...
4,12,guadiana,burdalo,79.0,0,-1.112043,42.606125,jucar,riu amadorio,None,None,None,caceres,extremadura,None,320.65,NaN,None


In [ ]:
def plot_all_reservoirs(reservoirs_list, with_transfers=False, legend=True):
    id_to_capacity = reservoirs_pd.set_index('id')['capacity'].to_dict()
    difference = {}
    transfers_log = pd.DataFrame()
    if with_transfers:
        
        _, transfers_log = make_optimal_transfers(reservoirs_list)

        for _, row in transfers_log.iterrows():
            if row['donor'] not in difference:
                difference[row['donor']] = 0
            difference[row['donor']] -= row['difference']
            if row['receiver'] not in difference:
                difference[row['receiver']] = 0
            difference[row['receiver']] += row['difference']

    next_year_predictions_path = PATHS['processed_data_notebooks'] / 'next_year_predictions.parquet'
    next_year_predictions = pd.read_parquet(next_year_predictions_path)

    water_pd['capacity'] = water_pd['id'].map(id_to_capacity)

    plt.figure(figsize=(12, 6))

    for reservoir in reservoirs_list:
        reservoir_pd = water_pd[water_pd['id'] == reservoir]
        reservoir_pd.set_index('date', inplace=True)
        df1 = reservoir_pd['storage'][-52*4:]/id_to_capacity[reservoir]
        df2 = (next_year_predictions[reservoir] + difference.get(reservoir, 0))/id_to_capacity[reservoir]
        df = pd.concat([df1, df2], axis=0)
        
        plt.plot(df.index,
                df,
                label=f'Reservoir {reservoir}')
        
    plt.axvline(x=df1.index[-1], color='k', linestyle='--', label='Next Year Prediction')
    plt.xlabel('Date')
    plt.ylabel('Storage (%)')
    plt.title('Evolution of Reservoir Storage')
    if legend:
        plt.legend()
    plt.show()
    return transfers_log

In [ ]:
reservoirs_list = reservoirs_pd[reservoirs_pd['province']=='huesca']['id'].values
reservoirs_list

In [ ]:
transfers_log = plot_all_reservoirs(reservoirs_list, with_transfers=False, legend=False)

In [ ]:
transfers_log = plot_all_reservoirs(reservoirs_list, with_transfers=True, legend=False)

In [ ]:
def plot_all_reservoirs_involved(reservoirs_list, with_transfers=False, legend=True):
    id_to_capacity = reservoirs_pd.set_index('id')['capacity'].to_dict()
    difference = {}
    _, transfers_log = make_optimal_transfers(reservoirs_list)

    for _, row in transfers_log.iterrows():
            if row['donor'] not in difference:
                difference[row['donor']] = 0
            difference[row['donor']] -= row['difference']
            if row['receiver'] not in difference:
                difference[row['receiver']] = 0
            difference[row['receiver']] += row['difference']
        
    reservoirs_involved = difference.keys()
    if not with_transfers:
         difference = {}

    next_year_predictions_path = PATHS['processed_data_notebooks'] / 'next_year_predictions.parquet'
    next_year_predictions = pd.read_parquet(next_year_predictions_path)

    water_pd['capacity'] = water_pd['id'].map(id_to_capacity)

    plt.figure(figsize=(12, 6))

    for reservoir in reservoirs_involved:
        reservoir_pd = water_pd[water_pd['id'] == reservoir]
        reservoir_pd.set_index('date', inplace=True)
        df1 = reservoir_pd['storage'][-52*4:]/id_to_capacity[reservoir]
        df2 = (next_year_predictions[reservoir] + difference.get(reservoir, 0))/id_to_capacity[reservoir]
        df = pd.concat([df1, df2], axis=0)
        
        plt.plot(df.index,
                df,
                label=f'Reservoir {int(reservoir)}')
        
    plt.axvline(x=df1.index[-1], color='k', linestyle='--', label='Next Year Prediction')
    plt.xlabel('Date')
    plt.ylabel('Storage (%)')
    plt.title('Evolution of Reservoir Storage')
    if legend:
        plt.legend()
    plt.show()
    return transfers_log

In [ ]:
plot_all_reservoirs_involved(reservoirs_list, with_transfers=False, legend=True)

In [ ]:
plot_all_reservoirs_involved(reservoirs_list, with_transfers=True, legend=True)